# Experiments 3 — Language-conditioned structural part editing

**执行顺序：Cell 1 → Cell 9。** 复用 `experiments2` 的原始 877 个共同有效对象，不重新生成三个 representation、不读 PartNet mesh、不重新校准 GT contact。

**两层分开报告**：
- **语言理解**：使用同一个 DeepSeek parser 将每条 controlled English edit instruction 映射为固定 `part_id + local axis + scale factor`。失败与错误目标都被记录；不借助 GT 提示重试。
- **Representation comparison**：对同一条*预先冻结的预期编辑操作*，按各自原生参数更新几何；对比独立 LLM 布局和相同 Anchor 初始布局两种条件。这样不让解释错误偏向任何 representation，也不把条件于解析正确的结果伪装成全样本表现。

**主任务**：对一个有直接子部件的**geometry-bearing parent**，通过自然语言要求沿本地 x/y/z 轴增加 25% 或缩小 20%；其他 part size 和共享结构不变。**主要指标**：该 parent 的*所有 outgoing shared edges* 的 OBB SAT margin drift；不是 exact mesh attachment。对 Global/Local 使用**原生参数更新（没有额外 attachment solver）**。

**准备**：将随包的 `methods.py`（含上一阶段的 patch）和本 notebook 放到项目目录；也可以只把 `methods3_append.py` 的内容追加到你当前完成 stage2 的 `methods.py` 末尾。旧 `partnet_results` 必须完整保留（特别是 `shared_specs.json`、`evaluation_all.json`、`generation/`、`same_geometry/objects/`）。

## Cell 1 — Import / reload

In [1]:
import os
import json
import importlib
import pandas as pd
import methods
from IPython.display import display
importlib.reload(methods)
from methods import *

## Cell 2 — Config （唯一需要改路径的地方）
`cfg.allow_api=True` 只用于 Cell 4 的一次共享语言解析；Cell 5–9 不调用 LLM。费用取决于之前没有缓存的指令数。

In [2]:
cfg = PartNetConfig()
cfg.project_dir = "."
cfg.output_dir = "partnet_results"
cfg.model = os.getenv("DEEPSEEK_MODEL", "deepseek-chat")
cfg.temperature = 0.0
cfg.max_tokens = 8192
cfg.workers = 4
cfg.allow_api = True
cfg.bootstrap_samples = 10000
cfg.show_every = 40
ed = EditingConfig()
ed.use_llm_parser = True
ed.parser_attempts = 2
ed.output_subdir = "experiments3"
print("原结果目录:", cfg.output_path().resolve())
print("新实验目录:", cfg.output_path(ed.output_subdir).resolve())

原结果目录: D:\MyJupyter\Works\ASPT\partnet_results
新实验目录: D:\MyJupyter\Works\ASPT\partnet_results\experiments3


## Cell 3 — 从冻结结构构造受控语言编辑任务（离线）
任务只依赖 shared parts/connections；不以实验结果或 GT adjacency 选 target。

In [3]:
edit_tasks = build_experiments3_edit_tasks(cfg, ed)
print("对象/任务:", edit_tasks["object_count"])
for task in edit_tasks["tasks"][:5]:
    print(task["object_id"], task["instruction"])

编辑任务固定：877 条（内部 target=508）。
对象/任务: 877
2472 Increase the depth of the seat single surface by 25% along its local y-axis. Keep the sizes of all other parts unchanged and preserve the existing attachments.
2653 Increase the height of the back support by 25% along its local z-axis. Keep the sizes of all other parts unchanged and preserve the existing attachments.
2694 Increase the width of the seat support by 25% along its local x-axis. Keep the sizes of all other parts unchanged and preserve the existing attachments.
2787 Decrease the depth of the back support by 20% along its local y-axis. Keep the sizes of all other parts unchanged and preserve the existing attachments.
2889 Decrease the height of the back support by 20% along its local z-axis. Keep the sizes of all other parts unchanged and preserve the existing attachments.


## Cell 4 — 一次性共享 LLM 语言解析（支持断点续跑）
仅验证 JSON schema 的失败可重试。选错 target/axis/scale 是被测的语言理解错误，不把 gold 反馈给 LLM。输出的 plan 不分支生成。

In [4]:
parse_results = run_experiments3_language_parser(edit_tasks, cfg, ed)
print("valid schema:", parse_results["valid_schema"], "/", parse_results["total"])
print("correct intent:", parse_results["correct"], "/", parse_results["total"])
print("parse error examples:")
for row in [r for r in parse_results["rows"] if not r["parse_correct"]][:8]:
    print(row["object_id"], row["status"], row["parsed_plan"])

Language edit parsing: 40/877
Language edit parsing: 80/877
Language edit parsing: 120/877
Language edit parsing: 160/877
Language edit parsing: 200/877
Language edit parsing: 240/877
Language edit parsing: 280/877
Language edit parsing: 320/877
Language edit parsing: 360/877
Language edit parsing: 400/877
Language edit parsing: 440/877
Language edit parsing: 480/877
Language edit parsing: 520/877
Language edit parsing: 560/877
Language edit parsing: 600/877
Language edit parsing: 640/877
Language edit parsing: 680/877
Language edit parsing: 720/877
Language edit parsing: 760/877
Language edit parsing: 800/877
Language edit parsing: 840/877
Language edit parsing: 877/877
语言解析：正确 855/877，schema valid 877/877。
valid schema: 877 / 877
correct intent: 855 / 877
parse error examples:
38596 valid {'target_part_id': 'part_19', 'axis': 'x', 'scale_factor': 0.8}
42941 valid {'target_part_id': 'part_8', 'axis': 'x', 'scale_factor': 0.8}
45056 valid {'target_part_id': 'part_19', 'axis': 'z', 'sca

## Cell 5 — 三表示语言编辑几何评估（完全离线）
对原主实验独立生成的布局，以及从 Anchor **同一初始几何重编码**的布局，都使用同一意图计划和各自原生 resize rule。原始 generation 和 experiments2 的结果不会改动。

In [5]:
cfg.allow_api = False
edit_geometry = run_experiments3_edit_evaluation(edit_tasks, cfg, ed)
print("geometry cohort:", edit_geometry["object_count"])
print("method-object-edit records:", len(edit_geometry["records"]))

语言编辑几何执行：40/877
语言编辑几何执行：80/877
语言编辑几何执行：120/877
语言编辑几何执行：160/877
语言编辑几何执行：200/877
语言编辑几何执行：240/877
语言编辑几何执行：280/877
语言编辑几何执行：320/877
语言编辑几何执行：360/877
语言编辑几何执行：400/877
语言编辑几何执行：440/877
语言编辑几何执行：480/877
语言编辑几何执行：520/877
语言编辑几何执行：560/877
语言编辑几何执行：600/877
语言编辑几何执行：640/877
语言编辑几何执行：680/877
语言编辑几何执行：720/877
语言编辑几何执行：760/877
语言编辑几何执行：800/877
语言编辑几何执行：840/877
语言编辑几何执行：877/877
geometry cohort: 877
method-object-edit records: 5262


## Cell 6 — 统计：object-level paired / category bootstrap / Holm
原始任务 cohort 与“解析正确 subset”分开报告，不人为剔除复杂对象。

In [6]:
edit_results = summarize_experiments3_edit_results(edit_tasks, parse_results, edit_geometry, cfg, ed)
print("parser accuracy:", edit_results["parser_success"], "/", edit_results["cohort_size"])
print("statistics:", len(edit_results["paired_statistics"]))

Experiments3：877 objects；解析正确 855/877；两种初始布局条件统计已保存。
parser accuracy: 855 / 877
statistics: 24


## Cell 7 — 主表：连接保持和副作用（两种初始布局）
`outgoing_margin_drift` 越低越好。`target_scale_factor_error`、`untargeted_max_size_drift` 是操作约束检查，通常由代码保证为 0，不应作为 Anchor 的独特优势。

In [7]:
main_rows = [r for r in edit_results["summary"] if r["subset"] == "intended_plan_full"]
main_df = pd.DataFrame(main_rows)
selected_metrics = ["outgoing_margin_drift", "outgoing_after_abs_margin", "outgoing_contact_proxy_rate", "incoming_margin_drift", "outside_subtree_max_center_drift", "untargeted_max_size_drift", "target_scale_factor_error"]
display(main_df[main_df["metric"].isin(selected_metrics)].pivot_table(index=["regime", "method"], columns="metric", values="mean", aggfunc="first").round(6))
primary_stats = [r for r in edit_results["paired_statistics"] if r["subset"] == "intended_plan_full" and r["metric"] == "outgoing_margin_drift"]
columns = ["regime", "baseline", "n", "anchor_mean", "baseline_mean", "mean_improvement", "ci_low", "ci_high", "holm_p", "wins", "ties", "losses"]
display(pd.DataFrame(primary_stats)[columns].round(6))

metric                         incoming_margin_drift  \
regime                 method                          
independent_generation anchor               0.000668   
                       global               0.009087   
                       local                0.009644   
same_initial_geometry  anchor               0.000668   
                       global               0.009198   
                       local                0.009198   

metric                         outgoing_after_abs_margin  \
regime                 method                              
independent_generation anchor                   0.026007   
                       global                   0.088130   
                       local                    0.080414   
same_initial_geometry  anchor                   0.026007   
                       global                   0.033657   
                       local                    0.033657   

metric                         outgoing_contact_proxy_rate  \
regime                 method                                
independent_generation anchor                     0.787721   
                       global                     0.305536   
                       local                      0.329357   
same_initial_geometry  anchor                     0.787721   
                       global                     0.535073   
                       local                      0.535073   

metric                         outgoing_margin_drift  \
regime                 method                          
independent_generation anchor               0.000853   
                       global               0.011619   
                       local                0.010867   
same_initial_geometry  anchor               0.000853   
                       global               0.010345   
                       local                0.010345   

metric                         outside_subtree_max_center_drift  \
regime                 method                                     
independent_generation anchor                               0.0   
                       global                               0.0   
                       local                                0.0   
same_initial_geometry  anchor                               0.0   
                       global                               0.0   
                       local                                0.0   

metric                         target_scale_factor_error  \
regime                 method                              
independent_generation anchor                        0.0   
                       global                        0.0   
                       local                         0.0   
same_initial_geometry  anchor                        0.0   
                       global                        0.0   
                       local                         0.0   

metric                         untargeted_max_size_drift  
regime                 method                             
independent_generation anchor                        0.0  
                       global                        0.0  
                       local                         0.0  
same_initial_geometry  anchor                        0.0  
                       global                        0.0  
                       local                         0.0

,regime,baseline,n,anchor_mean,baseline_mean,mean_improvement,ci_low,ci_high,holm_p,wins,ties,losses
0,independent_generation,global,877,0.000853,0.011619,0.010766,0.009605,0.011972,0.0,392,442,43
1,independent_generation,local,877,0.000853,0.010867,0.010013,0.008899,0.011148,0.0,396,444,37
2,same_initial_geometry,global,877,0.000853,0.010345,0.009492,0.008470,0.010531,0.0,347,529,1
3,same_initial_geometry,local,877,0.000853,0.010345,0.009492,0.008470,0.010531,0.0,347,529,1


## Cell 8 — 类别差异和 end-to-end 解析正确 subset

In [8]:
category_rows = [r for r in edit_results["category_summary"] if r["subset"] == "intended_plan_full" and r["metric"] == "outgoing_margin_drift"]
display(pd.DataFrame(category_rows).pivot(index=["category", "regime"], columns="method", values="mean").round(6))
conditional = [r for r in edit_results["paired_statistics"] if r["subset"] == "correctly_parsed_subset" and r["metric"] == "outgoing_margin_drift"]
if conditional:
    display(pd.DataFrame(conditional)[columns].round(6))
else:
    print("尚无解析正确样本；条件分析为空，完整 intended-plan 分析仍然有效。")
relevance_rows = [r for r in edit_results["axis_relevance_summary"] if r["subset"] == "intended_plan_full" and r["metric"] == "outgoing_margin_drift"]
print("Attachment-axis sensitivity (descriptive, not the primary test):")
display(pd.DataFrame(relevance_rows).pivot(index=["regime", "axis_relevance_group"], columns="method", values="mean").round(6))

method                                     anchor    global     local
category         regime                                              
Bottle           independent_generation  0.000231  0.009316  0.009038
                 same_initial_geometry   0.000231  0.008939  0.008939
Chair            independent_generation  0.000173  0.009010  0.004679
                 same_initial_geometry   0.000173  0.008139  0.008139
Faucet           independent_generation  0.000156  0.013655  0.012747
                 same_initial_geometry   0.000156  0.015253  0.015253
Knife            independent_generation  0.000000  0.002639  0.002717
                 same_initial_geometry   0.000000  0.002107  0.002107
Lamp             independent_generation  0.000266  0.010257  0.011210
                 same_initial_geometry   0.000266  0.012096  0.012096
Laptop           independent_generation  0.000000  0.001959  0.001822
                 same_initial_geometry   0.000000  0.001979  0.001979
Mug              independent_generation  0.001542  0.034741  0.033610
                 same_initial_geometry   0.001542  0.021269  0.021269
Scissors         independent_generation  0.001461  0.002888  0.005290
                 same_initial_geometry   0.001461  0.003683  0.003683
StorageFurniture independent_generation  0.001054  0.011841  0.010469
                 same_initial_geometry   0.001054  0.012786  0.012786
Table            independent_generation  0.000056  0.009449  0.006171
                 same_initial_geometry   0.000056  0.007246  0.007246
Vase             independent_generation  0.004422  0.021964  0.021569
                 same_initial_geometry   0.004422  0.020208  0.020208

,regime,baseline,n,anchor_mean,baseline_mean,mean_improvement,ci_low,ci_high,holm_p,wins,ties,losses
0,independent_generation,global,855,0.000866,0.011662,0.010796,0.009587,0.012021,0.0,380,436,39
1,independent_generation,local,855,0.000866,0.010729,0.009863,0.008714,0.011014,0.0,384,436,35
2,same_initial_geometry,global,855,0.000866,0.010336,0.009470,0.008413,0.010546,0.0,337,517,1
3,same_initial_geometry,local,855,0.000866,0.010336,0.009470,0.008413,0.010546,0.0,337,517,1


Attachment-axis sensitivity (descriptive, not the primary test):


method                                             anchor    global     local
regime                 axis_relevance_group                                  
independent_generation attachment_axis_active    0.000130  0.018727  0.017828
                       attachment_axis_inactive  0.001438  0.005874  0.005241
same_initial_geometry  attachment_axis_active    0.000130  0.019447  0.019447
                       attachment_axis_inactive  0.001438  0.002989  0.002989

## Cell 9 — 打包本轮结果给我（无需上传庞大的 raw generation 缓存）

In [9]:
from zipfile import ZipFile, ZIP_DEFLATED
files_to_share = ["edit_tasks.json", "language_parse_summary.json", "final_results.json", "geometry_results.json"]
review_path = cfg.output_path(f"{ed.output_subdir}/experiments3_review.zip")
with ZipFile(review_path, "w", compression=ZIP_DEFLATED) as archive:
    for filename in files_to_share:
        path = cfg.output_path(f"{ed.output_subdir}/{filename}")
        if not path.is_file():
            raise FileNotFoundError(path)
        archive.write(path, arcname=filename)
print("上传:", review_path.resolve())
print("文件大小 (MB):", round(review_path.stat().st_size / 1024**2, 2))

上传: D:\MyJupyter\Works\ASPT\partnet_results\experiments3\experiments3_review.zip
文件大小 (MB): 0.18
